# Pipeline de Co-retweet — execução ponta a ponta

Roda os seis módulos da pipeline sobre **todos os CSVs de um evento** e gera o grafo
de comunidades. Para trocar o evento, altere `EVENTO` na célula de configuração.

Etapas: **M1** carga → **M2** filtragem → **M3** matriz bipartida → **M4** projeção Jaccard
→ **M5** backbone (τ) → **M6** comunidades (Leiden). Cada etapa persiste seus artefatos em
`data/processed/<evento>/` e tem uma célula de inspeção preliminar.

In [1]:
import sys
from pathlib import Path

# Raiz do projeto: o notebook vive em notebooks/, mas pode rodar de notebooks/
# (Jupyter) ou da raiz do repositório (nbconvert/CI).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))  # habilita `from modules.* import ...`

import json
import numpy as np
import pandas as pd
import scipy.sparse as sp

from modules.load import RetweetLoader
from modules.filter import NoiseFilter
from modules.bipartite import BipartiteBuilder
from modules.project import JaccardProjector
from modules.backbone import BackboneExtractor
from modules.community import CommunityDetector

## Configuração

Troque `EVENTO` para rodar a pipeline em outro conjunto de dados. Os parâmetros
em aberto (`N`, `τ`) devem ser calibrados em `notebooks/exploratory-analysis.ipynb`
antes de rodar em escala.

In [ ]:
# ---- TROQUE AQUI O EVENTO ----
# Opções (subpastas de data/raw/): "invasao-3-poderes", "eleicoes",
#                                   "roberto-jefferson", "mobilizacao-0709"
EVENTO = "invasao-3-poderes"

# ---- Parâmetros da pipeline (calibrar antes de rodar em escala) ----
MIN_USER_RETWEETS = 3        # N  — filtro de atividade (mín. de retweets por usuário)
TAU = 0.10                   # τ  — peso Jaccard mínimo no backbone
RESOLUTION = 1.0             # resolução do Leiden

RAW_DIR = PROJECT_ROOT / "data" / "raw" / EVENTO
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / EVENTO
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(), f"Pasta do evento não encontrada: {RAW_DIR}"
csvs = sorted(RAW_DIR.glob("*.csv"))
print(f"Evento: {EVENTO}")
print(f"Raw:       {RAW_DIR}  ({len(csvs)} CSVs)")
for c in csvs:
    print(f"  - {c.name}")
print(f"Processed: {PROCESSED_DIR}")
print(f"Parâmetros: N={MIN_USER_RETWEETS}, τ={TAU}, resolution={RESOLUTION}")

## Módulo 1 — Carga de retweets

Lê todos os CSVs do evento e mantém apenas os retweets.

In [3]:
loader = RetweetLoader(RAW_DIR)        # diretório -> carrega todos os *.csv
retweets = loader.load()
loader.save(retweets, PROCESSED_DIR)   # -> retweets.parquet
print(f"Retweets carregados: {len(retweets):,}")
retweets.head()

Retweets carregados: 1,100,026


,author_id,referenced_tweet_id,created_at
0,1146896004341489675,1611790880922341376,2023-01-08 09:00:02
1,713194373412954112,1611938142445076483,2023-01-08 09:00:03
2,1034053475867533312,1611922426308296704,2023-01-08 09:00:04
3,1567983664423735301,1612009274473074688,2023-01-08 09:00:05
4,414776107,1611901031058743301,2023-01-08 09:00:13


In [4]:
# Inspeção M1
print(f"Linhas (retweets):  {len(retweets):,}")
print(f"Usuários únicos:    {retweets['author_id'].nunique():,}")
print(f"Tweets únicos:      {retweets['referenced_tweet_id'].nunique():,}")
print(f"Período:            {retweets['created_at'].min()}  →  {retweets['created_at'].max()}")

Linhas (retweets):  1,100,026
Usuários únicos:    321,268
Tweets únicos:      29,206
Período:            2023-01-08 09:00:02  →  2023-01-09 08:59:59


## Módulo 2 — Filtragem de ruído

Remove usuários inativos (< N retweets). Tweets virais são preservados (ver D5).

In [ ]:
nf = NoiseFilter(min_user_retweets=MIN_USER_RETWEETS)
filtered = nf.apply(retweets)
nf.save(filtered, PROCESSED_DIR)       # -> filtered_retweets.parquet + filter_stats.json
print(f"Retweets após filtro: {len(filtered):,}")
filtered.head()

In [6]:
# Inspeção M2
print(json.dumps(nf.stats, indent=2, ensure_ascii=False))
ret = len(filtered) / len(retweets) if len(retweets) else 0
print(f"\nRetenção de linhas: {len(retweets):,} -> {len(filtered):,} ({ret:.1%})")

{
  "initial": {
    "rows": 1100026,
    "users": 321268,
    "tweets": 29206
  },
  "after_user_filter": {
    "rows": 813100,
    "users": 85087,
    "tweets": 24104
  },
  "after_viral_filter": {
    "rows": 813100,
    "users": 85087,
    "tweets": 24104
  }
}

Retenção de linhas: 1,100,026 -> 813,100 (73.9%)


## Módulo 3 — Matriz bipartida usuário × tweet

In [3]:
if PROCESSED_DIR.exists():
  bg = BipartiteBuilder().load(PROCESSED_DIR)
else:
  bg = BipartiteBuilder().build(filtered)
  bg.save(PROCESSED_DIR)                  # -> bipartite_B.npz + índices
print("Matriz B:", bg.B.shape, "| nnz:", f"{bg.B.nnz:,}")

Matriz B: (85087, 24104) | nnz: 813,089


In [4]:
# Inspeção M3
n_u, n_t = bg.B.shape
print(f"Usuários (linhas): {n_u:,}")
print(f"Tweets (colunas):  {n_t:,}")
print(f"Não-zeros:         {bg.B.nnz:,}")
print(f"Densidade:         {bg.B.nnz / (n_u * n_t):.2e}")
col_sum = np.asarray(bg.B.sum(axis=0)).ravel()   # retweetadores por tweet sobrevivente
print(f"Máx retweetadores num tweet sobrevivente: {int(col_sum.max()):,}")

Usuários (linhas): 85,087
Tweets (colunas):  24,104
Não-zeros:         813,089
Densidade:         3.96e-04
Máx retweetadores num tweet sobrevivente: 9,201


## Pré-voo da projeção (custo de memória)

⚠️ O custo da projeção `B·Bᵀ` cresce com o **quadrado** do nº de retweetadores do tweet
mais denso. Como os tweets virais são preservados (D5), o número de pares co-retweet pode
ser grande demais para materializar a matriz densa de uma vez — por isso a projeção é feita
em blocos, com corte do peso Jaccard durante a geração. Rode a célula abaixo **antes** do
Módulo 4 para dimensionar o custo.

In [ ]:
# Pré-voo: limite superior de pares co-retweet (proxy do custo da projeção)
from math import comb

pair_upper = int(sum(comb(int(k), 2) for k in col_sum if k >= 2))
print(f"Limite superior de pares co-retweet: {pair_upper:,}")
if pair_upper > 50_000_000:
    print("\n⚠️  ALTO: materializar B·Bᵀ inteiro estouraria a memória.")
    print("    A projeção em blocos com corte de Jaccard (Módulo 4) mantém o uso")
    print("    de memória limitado e só persiste as arestas acima do limiar.")
else:
    print("OK: custo da projeção dentro do tratável.")

## Módulo 4 — Projeção Jaccard (grafo de co-retweet)

In [6]:
pg = JaccardProjector().project(bg)
pg.save(PROCESSED_DIR, prefix="projection")   # -> projection_W.npz + índice
print("Grafo projetado:", pg.W.shape, "| arestas:", f"{pg.W.nnz:,}")

: 

In [ ]:
# Inspeção M4 — distribuição dos pesos Jaccard (ajuda a escolher τ)
w = pg.W.data
print(f"Nós:     {pg.W.shape[0]:,}")
print(f"Arestas: {pg.W.nnz:,}")
print(f"Peso Jaccard — min/méd/máx: {w.min():.3f} / {w.mean():.3f} / {w.max():.3f}")
print("\nArestas retidas por limiar τ:")
for t in (0.05, 0.10, 0.15, 0.20):
    print(f"  τ >= {t:.2f}:  {int((w >= t).sum()):,}  ({(w >= t).mean():.1%})")

## Módulo 5 — Backbone (universal threshold τ)

In [ ]:
bb = BackboneExtractor(tau=TAU)
pg_bb = bb.extract(pg)
bb.save_stats(PROCESSED_DIR)                   # -> backbone_stats.json
pg_bb.save(PROCESSED_DIR, prefix="backbone")   # -> backbone_W.npz + índice
print("Backbone:", pg_bb.W.shape, "| arestas:", f"{pg_bb.W.nnz:,}")

In [ ]:
# Inspeção M5
print(json.dumps(bb.stats, indent=2))
print(f"\nNós:     {bb.stats['before']['nodes']:,} -> {bb.stats['after']['nodes']:,}")
print(f"Arestas: {bb.stats['before']['edges']:,} -> {bb.stats['after']['edges']:,}")

## Módulo 6 — Detecção de comunidades (Leiden)

In [ ]:
cr = CommunityDetector(resolution=RESOLUTION).detect(pg_bb)
cr.save(PROCESSED_DIR)                          # -> community_graph.graphml + membership.parquet
print(f"Comunidades: {len(set(cr.membership))}")
print(f"Modularidade: {cr.partition.modularity:.4f}")

In [ ]:
# Inspeção M6
sizes = pd.Series(cr.membership).value_counts()
print(f"Nº de comunidades: {len(sizes)}")
print(f"Modularidade:      {cr.partition.modularity:.4f}")
print(f"Nós no grafo:      {cr.g.vcount():,} | arestas: {cr.g.ecount():,}")
print("\nTamanho das 15 maiores comunidades:")
print(sizes.head(15).to_string())
if sizes.sum():
    print(f"\nTop 5 comunidades cobrem {sizes.head(5).sum() / sizes.sum():.1%} dos nós")

## Artefatos salvos

In [ ]:
print(f"Arquivos em {PROCESSED_DIR}:\n")
for f in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {f.name:34s} {f.stat().st_size / 1024:>10,.1f} KB")